In [0]:
from pyspark.sql.functions import *

df_category = spark.table("ecommerce_dev.bronze.category_translation")

df_category.printSchema()
display(df_category.limit(10))
print(f"Total rows: {df_category.count()}")

# Null counts on both category name columns
df_category.select(
    count(when(col("product_category_name").isNull(), True)).alias("null_pt"),
    count(when(col("product_category_name_english").isNull(), True)).alias("null_en"),
    count(when(col("product_category_name").isNull() & col("product_category_name_english").isNull(), True)).alias("null_both")
).show()

# Duplicate check on product_category_name (expected PK)
df_category.groupBy("product_category_name").count().filter("count > 1").show()

root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_table: string (nullable = true)



product_category_name,product_category_name_english,_rescued_data,_ingested_at,_source_file,_source_table
beleza_saude,health_beauty,null,2026-08-04T00:56:12.497Z,/Volumes/ecommerce_dev/raw_data/landing/product_category_name_translation.csv,category_translation
informatica_acessorios,computers_accessories,null,2026-08-04T00:56:12.497Z,/Volumes/ecommerce_dev/raw_data/landing/product_category_name_translation.csv,category_translation
automotivo,auto,null,2026-08-04T00:56:12.497Z,/Volumes/ecommerce_dev/raw_data/landing/product_category_name_translation.csv,category_translation
cama_mesa_banho,bed_bath_table,null,2026-08-04T00:56:12.497Z,/Volumes/ecommerce_dev/raw_data/landing/product_category_name_translation.csv,category_translation
moveis_decoracao,furniture_decor,null,2026-08-04T00:56:12.497Z,/Volumes/ecommerce_dev/raw_data/landing/product_category_name_translation.csv,category_translation
esporte_lazer,sports_leisure,null,2026-08-04T00:56:12.497Z,/Volumes/ecommerce_dev/raw_data/landing/product_category_name_translation.csv,category_translation
perfumaria,perfumery,null,2026-08-04T00:56:12.497Z,/Volumes/ecommerce_dev/raw_data/landing/product_category_name_translation.csv,category_translation
utilidades_domesticas,housewares,null,2026-08-04T00:56:12.497Z,/Volumes/ecommerce_dev/raw_data/landing/product_category_name_translation.csv,category_translation
telefonia,telephony,null,2026-08-04T00:56:12.497Z,/Volumes/ecommerce_dev/raw_data/landing/product_category_name_translation.csv,category_translation
relogios_presentes,watches_gifts,null,2026-08-04T00:56:12.497Z,/Volumes/ecommerce_dev/raw_data/landing/product_category_name_translation.csv,category_translation


Total rows: 71
+-------+-------+---------+
|null_pt|null_en|null_both|
+-------+-------+---------+
|      0|      0|        0|
+-------+-------+---------+

+---------------------+-----+
|product_category_name|count|
+---------------------+-----+
+---------------------+-----+



In [0]:
df_products_silver = spark.table("ecommerce_dev.silver.products")

orphan_categories = (df_products_silver
    .select("product_category_name")
    .distinct()
    .join(df_category.select("product_category_name"), on="product_category_name", how="left_anti")
)

print(f"Product categories with no translation match: {orphan_categories.count()}")
display(orphan_categories)

Product categories with no translation match: 3


product_category_name
pc_gamer
portateis_cozinha_e_preparadores_de_alimentos
unknown


In [0]:
from pyspark.sql.functions import *

# --- Load from Bronze ---
df_category = spark.table("ecommerce_dev.bronze.category_translation")

In [0]:
# --- Transform: drop metadata, rename ingestion timestamp ---
df_silver_category = (df_category
    .withColumnRenamed("_ingested_at", "bronze_ingested_at")
    .drop("_rescued_data", "_source_file", "_source_table")
)

In [0]:
# --- Backfill missing categories found in products but absent from the translation file ---
backfill_rows = [
    ("pc_gamer", "pc_gamer", None, None, None),
    ("portateis_cozinha_e_preparadores_de_alimentos", "kitchen_portables_food_preparers", None, None, None),
    ("unknown", "unknown", None, None, None),
]

In [0]:
backfill_schema = "product_category_name string, product_category_name_english string, dummy1 string, dummy2 string, dummy3 string"
df_backfill = (spark.createDataFrame(backfill_rows, schema=backfill_schema)
    .drop("dummy1", "dummy2", "dummy3")
    .withColumn("bronze_ingested_at", lit(None).cast("timestamp"))
    .withColumn("is_backfilled", lit(True))
)

In [0]:
df_silver_category = (df_silver_category
    .withColumn("is_backfilled", lit(False))
    .unionByName(df_backfill)
)

In [0]:
df_silver_category = (df_silver_category
    .withColumn("is_backfilled", lit(False))
    .unionByName(df_backfill)
)

In [0]:
dupes = df_silver_category.groupBy("product_category_name").count().filter("count > 1")
print(f"Duplicate product_category_name in final Silver table: {dupes.count()}")
print(f"Total rows: {df_silver_category.count()}")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7916359624965407>, line 2
      1 dupes = df_silver_category.groupBy("product_category_name").count().filter("count > 1")
----> 2 print(f"Duplicate product_category_name in final Silver table: {dupes.count()}")
      3 print(f"Total rows: {df_silver_category.count()}")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1930, in DataFrame._to_table(self)
   1928 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1929     query = self._plan.to_proto(sel

In [0]:
(df_silver_category.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_dev.silver.category_translation")
)

In [0]:
# --- Enforce constraints ---
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.category_translation
    ALTER COLUMN product_category_name SET NOT NULL
""")
spark.sql("""
    ALTER TABLE ecommerce_dev.silver.category_translation
    ADD CONSTRAINT pk_category_translation PRIMARY KEY (product_category_name)
""")

# --- Table comment ---
spark.sql("""
    COMMENT ON TABLE ecommerce_dev.silver.category_translation IS
    'PT-to-English category name lookup. Source file had complete data (no nulls) 
    for all 71 original rows. 3 rows backfilled (is_backfilled = true) to prevent 
    orphaned joins against silver.products: pc_gamer and 
    portateis_cozinha_e_preparadores_de_alimentos are real Olist categories missing 
    from the official translation file; unknown is the placeholder value used in 
    silver.products for null category names.'
""")

DataFrame[]

In [0]:
df_silver_category = spark.table("ecommerce_dev.silver.category_translation")

dupes = df_silver_category.groupBy("product_category_name").count().filter("count > 1")
print(f"Duplicate product_category_name in final Silver table: {dupes.count()}")
print(f"Total rows: {df_silver_category.count()}")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7916359624965358>, line 4
      1 df_silver_category = spark.table("ecommerce_dev.silver.category_translation")
      3 dupes = df_silver_category.groupBy("product_category_name").count().filter("count > 1")
----> 4 print(f"Duplicate product_category_name in final Silver table: {dupes.count()}")
      5 print(f"Total rows: {df_silver_category.count()}")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319     return table[0][0].as_py()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1930, in DataFrame._to_table(self)
   1928 def _to_table(self) -